# Becke Partition 一阶梯度简单理解

In [1]:
from pyscf import gto, dft, lib, grad, hessian, data
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
def get_grids(xyz):
    mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()
    grids = dft.grid.Grids(mol)
    grids.radii_adjust = dft.radi.becke_atomic_radii_adjust
    grids.build(sort_grids=False)
    return mol, grids

## PySCF 解析与数值导数验证

需要留意，PySCF 先前使用的 `grids_response_cc` 似乎是由于后来引入 padding 的问题，其结果我不太确定是否正确。目前使用的是 PySCF 中用于计算 VV10 导数所用到的函数。

In [4]:
xyz_0 = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""
xyz_p = """
N  0.0001   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""
xyz_m = """
N -0.0001   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

In [5]:
mol, grids = get_grids(xyz_0)

一阶解析格点导数计算如下：

In [6]:
dw_ref = hessian.rks.get_dweight_dA(mol, grids)
dw_ref.shape

(4, 3, 43328)

作为例子，一阶数值导数的第一个分量计算如下：

In [7]:
tmp_num_d = (get_grids(xyz_p)[1].weights - get_grids(xyz_m)[1].weights) / (2 * 0.0001 / data.nist.BOHR)

解析与数值导数有强一致。

In [8]:
np.allclose(dw_ref[0, 0], tmp_num_d)

True

## Becke Partition 一阶梯度实现与公式对应

### 零阶记号与程序重新定义

为了方便公式推导与程序实现，我们需要重新定义一些记号。

- Becke 1988 原文的角标 $i, j$ 现在通常是指占据轨道。我们使用 $A, B, M, N$ 表示原子指标。
- 我们在可能的情况下，优先使用矩阵或张量记号。

In [9]:
nonpad_mask = grids.atm_idx != -1
quadrature_weights = grids.quadrature_weights[nonpad_mask]
grid_coords = grids.coords[nonpad_mask]
atm_coords = mol.atom_coords()
atm_indices = grids.atm_idx[nonpad_mask]

natm = atm_coords.shape[0]
ngrids = grid_coords.shape[0]
becke_radii_adjust = dft.radi.becke_atomic_radii_adjust(mol, grids.atomic_radii)
radii_table = np.array([becke_radii_adjust(i, j, 0) for i in range(natm) for j in range(natm)]).reshape(natm, natm)

- `wquad` $w_g^\text{quad}$：原始 Lebedev 权重，维度 $(g,)$ `(ngrids,)`

- `a` $a_{AB}$：Becke radii 矫正表，维度 $(A, B)$ `(natm, natm)`

In [10]:
wquad = quadrature_weights
a = radii_table

- `atm_coords` $R_{A t}$：原子坐标，维度 $(A, 3)$ `(natm, 3)`

- `atom_dist` $\Vert R \Vert_{AB}$：原子间距离，维度 $(A, B)$ `(natm, natm)`；其只作为分母出现，对角元设为 $\infty$，避免除零

    $$
    \Vert R \Vert_{AB} = 
    \begin{cases}
    \sqrt{\sum_t (R_{B t} - R_{A t})^2} & A \neq B \\
    \infty & A = B
    \end{cases}
    $$

In [11]:
atom_dist = np.linalg.norm(atm_coords[:, None, :] - atm_coords[None, :, :], axis=-1)
for i in range(natm):
    atom_dist[i, i] = np.inf

- `grid_coords` $r_{g t}$：格点坐标，维度 $(g, 3)$ `(ngrids, 3)`

- `grid_dist` $\Vert r \Vert_{Ag}$：原子与格点间距离，维度 $(A, g)$ `(natm, ngrids)`

    $$
    \Vert r \Vert_{A g} = \sqrt{\sum_t (r_{g t} - R_{A t})^2}
    $$

In [12]:
grid_dist = np.linalg.norm(grid_coords[None, :, :] - atm_coords[:, None, :], axis=-1)

- `mu` $\mu_{MNg}$：椭球坐标差分量，维度 $(M, N, g)$ `(natm, natm, ngrids)`

    $$
    \mu_{M N g} = \frac{\Vert r \Vert_{M g} - \Vert r \Vert_{N g}}{\Vert R \Vert_{M N}} \tag{11}
    $$

In [13]:
mu = (grid_dist[:, None, :] - grid_dist[None, :, :]) / atom_dist[:, :, None]  # eq (11)

assert np.allclose(- mu.swapaxes(0, 1), mu)

- `s` $s_{M N g}$: Becke 特征函数，维度 $(M, N, g)$ `(natm, natm, ngrids)`，其定义为 (留意这里 $a_{M N}$ 不是格点或原子坐标依赖量，我们通常不将它作为变量看待)

    $$
    s_{M N g} = 
    \begin{cases}
    s(\mu_{M N g}, a_{M N}) & M \neq N \\
    1 & M = N
    \end{cases}
    $$

    这里用 lambda 函数实现，实际程序中还需要调整。这里需要明确函数关系 (自变量是 $\mu$ 还是 $\nu$)。

    自变量为 $\nu$ 的函数：

    $$
    \begin{align}
    p(\nu) &= \frac{3}{2} \nu - \frac{1}{2} \nu^3 \tag{19} \\
    f_3(\nu) &= p \circ p \circ p (\nu) \tag{20} \\
    s_3(\nu) &= \frac{1}{2} (1 - f_3(\nu)) \tag{21}
    \end{align}
    $$

     自变量为 $\mu$ 的函数：

    $$
    \begin{align}
    s (\mu) &= s_3 \circ \nu (\mu) \tag{A1} \\
    \nu(\mu) &= \mu + a ( 1 - \mu^2 ) \tag{A2}
    \end{align}
    $$

    同时注意到，后面计算过程中的乘积是不对 $M = N$ 的情形作的，因此 $M = N$ 的情形设置值为 1。

In [14]:
fn_p = lambda nu: 1.5 * nu - 0.5 * nu**3    # eq (19)
fn_f1 = fn_p
fn_f2 = lambda nu: fn_p(fn_f1(nu))
fn_f3 = lambda nu: fn_p(fn_f2(nu))          # eq (20)
fn_s3 = lambda nu: 0.5 * (1 - fn_f3(nu))    # eq (21)

fn_nu = lambda mu, a: mu + a * (1 - mu**2)  # eq (A2)
fn_s = lambda mu, a: fn_s3(fn_nu(mu, a))    # eq (A1)

In [15]:
s = fn_s(mu, a[:, :, None])
for M in range(natm):
    s[M, M] = 1

- `P` $P_{M g}$：Becke 权重，维度 $(M, g)$ `(natm, ngrids)`

    $$
    P_{M g} = \prod_{N \neq M} s_{M N g} \tag{13}
    $$

    当然，我们已经设置了 $s_{M M g} = 1$，因此 $N \neq M$ 的乘积条件，从程序实现角度，是可以省略的。

- `Z` $Z_g$：Becke 权重归一化因子，维度 $(g,)$ `(ngrids,)`

    $$
    Z_g = \sum_M P_{M g} \tag{22}
    $$

In [16]:
P = s.prod(axis=1)
Z = P.sum(axis=0)

- Pg $P^M_g$：格点所在原子的 Becke 权重，维度 $(g,)$ `(ngrids,)`

    $$
    P^M_g = P_{M g} \delta_{g \in M}
    $$

    需要留意，该张量的上标 $M$ 意味着 $g \in M$，但不参与张量权重。这个记号与通常的指标意义不同。

In [17]:
Pg = P[atm_indices, np.arange(ngrids)]

- `w` $w_g$：Becke 格点权重，维度 $(g,)$ `(ngrids,)`

    $$
    w_g = w_g^\text{quad} \frac{P^M_g}{Z_g} \tag{22}
    $$

In [18]:
w = wquad * Pg / Z

assert np.allclose(w, grids.weights[nonpad_mask])

### 格点坐标对原子导数的重要说明

我们在后续推导中，会认为格点坐标 $r_{gt}$ 是不依赖于原子坐标的。但作为代价，格点所对应的原子梯度的计算需要作特殊的全导数处理。

**从程序角度来说，这是错误的；但经过一些数学操作，这个问题是可以解决的**。实际上，当我们从一个偏移了坐标的原子开始，重新构建格点时，格点坐标是会随原子坐标的变化而变化的。因此，原则上当然要考虑格点坐标导数。如果忽略格点坐标导数，那么许多中间量的数值与解析导数是对不上的。

但引入格点坐标对原子导数的依赖后，梯度推导与计算会变得复杂。我们在这里开始，直到最后一步前，都忽略格点坐标的导数。我们没有办法核验中间量是否正确，但最终结果的数值与解析导数是一致的。

注意这不是非常 trivial 的做法。我们要分类讨论。

- 需要知道，格点 $r_g$ ($g \in M$) 是依赖于 $M$ 原子坐标的；但它不依赖于其他原子坐标。也就是说，对其他原子的偏导仍然是正确的。或者说，
  
    $$
    \frac{\partial w_g}{\partial R_{A t}} \quad (g \in M, A \neq M)
    $$

    这部分的偏导我们可以按照 $r_g$ 不依赖于原子核坐标的方式来计算，结果仍然是正确的。

- 对于 $g \in M, A = M$ 的情形，按照 $r_g$ 不依赖于原子核坐标的方式来计算偏导，结果是错误的。但我们可以利用全导数为零的特性，简化该计算为

    $$
    \frac{\partial w_g}{\partial R_{A t}} = - \sum_{M \neq A} \frac{\partial w_g}{\partial R_{M t}} \quad (g \in A)
    $$

对于上式的说明如下。首先，我们将原子核坐标与格点坐标同时考虑为变量：

$$
w_g (\bm{r}_g, \bm{R}_A, \bm{R}_B, \ldots)
$$

格点权重具有平移不变性，即对所有坐标增加一个相同的平移量 $\bm{t}$，格点权重不变：

$$
w_g (\bm{r}_g + \bm{t}, \bm{R}_A + \bm{t}, \bm{R}_B + \bm{t}, \ldots) - w_g (\bm{r}_g, \bm{R}_A, \bm{R}_B, \ldots) = 0
$$

对上式的等式左右作 $\bm{t}$ 的全导数，得到下式为零：

$$
\frac{\mathrm{d} w_g}{\mathrm{d} \bm{t}} = \frac{\partial w_g}{\partial \bm{r}_g} + \sum_M \frac{\partial w_g}{\partial \bm{R}_M} = 0
$$

现在考虑到 $g \in A$，并考虑如何求取对该原子核坐标 $\bm{R}_A$ 的梯度。首先我们知道，不论我们是否考虑了 $\bm{r}_g$ 是否依赖于 $\bm{R}_A$，其他 $M \neq A$ 的偏导数 $\frac{\partial w_g}{\partial \bm{R}_M}$ 都是正确的。因此我们可以将上式改写为：

$$
\frac{\partial w_g}{\partial \bm{R}_A} + \frac{\partial w_g}{\partial \bm{r}_g} = - \sum_{M \neq A} \frac{\partial w_g}{\partial \bm{R}_M}
$$

现在我们考虑对于 $g \in A$ 的情况 (即格点 $g$ 是原子 $A$ 所生成的 Lebedev 格点)。在实际程序中，$\bm{r}_g$ 随 $\bm{R}_A$ 是线性变化的；$\bm{R}_A$ 平移 $\bm{t}$ 后，$\bm{r}_g$ 也平移 $\bm{t}$；或者用数学的方式表述 (下述的单位矩阵是 3x3 的)，

$$
\frac{\mathrm{d} \bm{r}_g}{\mathrm{d} \bm{R}_A} = \mathbf{I}
$$

那么对 $w_g$ 求全导数：

$$
\frac{\mathrm{d} w_g}{\mathrm{d} \bm{R}_A} = \frac{\partial w_g}{\partial \bm{R}_A} + \frac{\partial w_g}{\partial \bm{r}_g} \mathbf{I} = - \sum_{M \neq A} \frac{\partial w_g}{\partial \bm{R}_M}
$$

格点 $g$ 所对应的原子 $A$ 是唯一必须要考虑全导数的项；其他原子的偏导数都是正确的。那么我们可以用所有 $M \neq A$ 原子的结果，反推 $A$ 原子的全导数。

In [19]:
assert np.isclose(np.abs(dw_ref.sum(axis=0)).max(), 0)

从现在开始，我们将忘记 $\bm{r}_g$ 对 $\bm{R}_A$ 有依赖这件事，直到最后结算 $w_g$ 的导数时再考虑。

### 椭球坐标差分量 $\mu_{M N g}$ 的导数

- $\Vert \partial r \Vert_{A t g}$ `dR_grid_dist` 格点距离导数，维度 $(A, t, g)$ `(natm, 3, ngrids)`：

    $$
    \Vert \partial r \Vert_{A t g} := \frac{\partial \Vert r \Vert_{A g}}{\partial R_{A t}} = \frac{R_{A t} - r_{g t}}{\Vert r \Vert_{A g}}
    $$

- $\Vert \partial R \Vert_{A B t}$ `dR_atom_dist` 原子距离导数，维度 $(A, B, t)$ `(natm, natm, 3)`：

    $$
    \Vert \partial R \Vert_{A B t} := \frac{\partial \Vert R \Vert_{AB}}{\partial R_{A t}} = \frac{R_{A t} - R_{B t}}{\Vert R \Vert_{AB}}
    $$

    需要留意，从定义上看，$\Vert \partial R \Vert_{A B t} = - \Vert \partial R \Vert_{B A t}$。对 B 原子的导数是反对称的。

In [20]:
dR_grid_dist = (atm_coords[:, :, None] - grid_coords.T[None, :, :]) / grid_dist[:, None, :]
dR_atom_dist = (atm_coords[:, None, :] - atm_coords[None, :, :]) / atom_dist[:, :, None]

assert np.allclose(- dR_atom_dist.swapaxes(0, 1), dR_atom_dist)

- $\partial \mu_{ABg} / \partial R_{A t} \; (\text{role A})$ `dR_mu_roleA` 椭球坐标差分量对左侧原子指标的导数，维度 $(A, B, t, g)$ `(natm, natm, 3, ngrids)`；

- $\partial \mu_{ABg} / \partial R_{B t} \; (\text{role B})$ `dR_mu_roleB` 椭球坐标差分量对右侧原子指标的导数，维度 $(A, B, t, g)$ `(natm, natm, 3, ngrids)`；

    $$
    \begin{align*}
    \frac{\partial \mu_{ABg}}{\partial R_{A t}} \; (\text{role A}) &= \frac{1}{\Vert R \Vert_{AB}} \big( \Vert \partial r \Vert_{A t g} - \mu_{A B g} \Vert \partial R \Vert_{A B t} \big) \\
    \frac{\partial \mu_{ABg}}{\partial R_{B t}} \; (\text{role B}) &= \frac{1}{\Vert R \Vert_{AB}} \big( - \Vert \partial r \Vert_{B t g} + \mu_{A B g} \Vert \partial R \Vert_{A B t} \big)
    \end{align*}
    $$

这里并非计算了真正的导数；真正的导数计算应该要按下式进行 (将会产生 5-dim 张量 $(A, M, N, t, g)$)：

$$
\frac{\partial \mu_{M N g}}{\partial R_{A t}} = \frac{\partial \mu_{A N g}}{\partial R_{A t}} \delta_{A M} + \frac{\partial \mu_{M A g}}{\partial R_{A t}} \delta_{A N}
$$

但上式中有很多的 delta，是非常稀疏的。我们在进行实际计算时，需要记得我们已经将一些 delta 函数提前缩并了。

同时，我们要注意到 role A 与 role B 的导数是反对称的；因此实际计算时我们只需要生成一个即可。

In [21]:
dR_mu_roleA = (  dR_grid_dist[:, None, :, :] - mu[:, :, None, :] * dR_atom_dist[:, :, :, None]) / atom_dist[:, :, None, None]
dR_mu_roleB = (- dR_grid_dist[None, :, :, :] + mu[:, :, None, :] * dR_atom_dist[:, :, :, None]) / atom_dist[:, :, None, None]

assert np.allclose(- dR_mu_roleB.swapaxes(0, 1), dR_mu_roleA)

### Becke 特征函数 $s(\mu)$ 导数

这里为了记号简便，我们直接用 $'$ 表示导数。但需要留意自变量是 $\mu$ 还是 $\nu$。

自变量为 $\nu$ 的函数：

$$
\begin{align*}
p' &= \frac{3}{2} (1 - \nu^2) \\
f_3' &= p' (f_2) p' (f_1) p' \\
s_3' &= - \frac{1}{2} f_3'
\end{align*}
$$

自变量为 $\mu$ 的函数：

$$
\begin{align*}
\nu' &= 1 - 2 a \mu \\
s' &= s_3' \nu'
\end{align*}
$$

In [22]:
fn_d_p = lambda nu: 1.5 * (1 - nu**2)
fn_d_f1 = fn_d_p
fn_d_f2 = lambda nu: fn_d_p(fn_f1(nu)) * fn_d_f1(nu)
fn_d_f3 = lambda nu: fn_d_p(fn_f2(nu)) * fn_d_f2(nu)
fn_d_s3 = lambda nu: -0.5 * fn_d_f3(nu)

fn_d_nu = lambda mu, a: 1 - 2 * a * mu
fn_d_s = lambda mu, a: fn_d_s3(fn_nu(mu, a)) * fn_d_nu(mu, a)

- $\frac{\partial s_{M N g}}{\partial \mu_{M N g}}$ `dmu_s` 特征函数导数，维度 $(M, N, g)$ `(natm, natm, ngrids)`

In [23]:
dmu_s = fn_d_s(mu, a[:, :, None])

最后，我们需要引入一个后续计算中常用的量 $\frac{\partial \log s_{M N g}}{\partial \mu_{M N g}}$ `dmu_log_s`，维度 $(M, N, g)$ `(natm, natm, ngrids)`

$$
\frac{\partial \log s_{M N g}}{\partial \mu_{M N g}} = \frac{1}{s_{M N g}} \frac{\partial s_{M N g}}{\partial \mu_{M N g}}
$$

其从程序实现角度来说，由于出现了 $s_{M N g}$ 出现在分母或 log 函数中，而非常小的 $s_{M N g}$ 会导致数值不稳定。我们会在计算中忽略过小的 $s_{M N g}$。同时，该量后续会用于求和，但求和条件是 $M \neq N$；因此我们会将原子指标上对角线的部分设置为 0。

In [24]:
s_safe_mask = np.abs(s) > 1e-14
s_safe = s.copy()
s_safe[~s_safe_mask] = 1.0
dmu_log_s = dmu_s / s_safe
for M in range(natm):
    dmu_log_s[M, M] = 0.0

### 权重函数 $P_{M g}$ 导数

首先回顾定义：

$$
P_{M g} = \prod_{N \neq M} s_{M N g} \tag{13}
$$

连乘积导数的通常技巧是对等式左右作对数，将连乘转为求和。

$$
\log P_{M g} = \sum_{N \neq M} \log s_{M N g}
$$

那么导数就可以写为

$$
\partial \log P_{M g} = \frac{1}{P_{M g}} \partial P_{M g} = \sum_{N \neq M} \partial \log s_{M N g}
$$

整理上式得到

$$
\partial P_{M g} = P_{M g} \sum_{N \neq M} \partial \log s_{M N g}
$$

现在我们要作实际的导数。注意到 $s$ 是 $\mu$ 单纯的函数，因此作为张量的 $s_{M N g}$ 也只对 $\mu_{M N g}$ 一一对应地有导数；换句话说，

$$
\frac{\partial s_{M N g}}{\partial \mu_{A B g}} = \frac{\partial s_{M N g}}{\partial \mu_{M N g}} \delta_{A M} \delta_{B N}
$$

依据链式法则，我们记导数 $\partial P_{M g} / \partial R_{A t}$ `dR_P`，维度 $(M, A, t, g)$ `(natm, natm, 3, ngrids)`：

$$
\frac{\partial P_{M g}}{\partial R_{A t}} = P_{M g} \sum_{N \neq M} \frac{\partial \log s_{M N g}}{\partial R_{A t}} = P_{M g} \sum_{N \neq M} \frac{\partial \log s_{M N g}}{\partial \mu_{M N g}} \frac{\partial \mu_{M N g}}{\partial R_{A t}}
$$

回顾到 $\mu_{M N g}$ 对原子核 $A$ 坐标的导数，有 role A 与 role B。

- Role A 的导数，即 $M = A$ 的情况：

    $$
    \frac{\partial P_{A g}}{\partial R_{A t}} = P_{A g} \sum_{N \neq A} \frac{\partial \log s_{A N g}}{\partial \mu_{A N g}} \frac{\partial \mu_{A N g}}{\partial R_{A t}} \quad (\text{role A})
    $$

    留意我们已经在程序中设置了 $\partial \log s_{A N g} / \partial \mu_{A N g}$ 在 $N = A$ 的情形的值为零，因此在程序实现时，上式可以直接对所有原子指标 $N$ 求和。

- Role B 的导数，即 $N = A$ 的情况，那么此时 $N$ 是不被求和而被限制为 $A$ 原子：

    $$
    \frac{\partial P_{M g}}{\partial R_{A t}} = P_{M g} \frac{\partial \log s_{M A g}}{\partial \mu_{M A g}} \frac{\partial \mu_{M A g}}{\partial R_{A t}} \quad (\text{role B}, M \neq A)
    $$

- 将 role A 与 role B 的导数合并，可以得到最终的导数。

In [25]:
dR_P_roleA = np.einsum("Ag, ANg, ANtg -> Atg", P, dmu_log_s, dR_mu_roleA)
dR_P_roleB = np.einsum("Mg, MAg, MAtg -> MAtg", P, dmu_log_s, dR_mu_roleB)
dR_P = dR_P_roleB.copy()
for A in range(natm):
    dR_P[A, A] = dR_P_roleA[A]

同时，我们也可以给出 Becke 权重归一化因子的导数 `dR_Z`，维度 $(A, t, g)$ `(natm, 3, ngrids)`：

$$
\frac{\partial Z_g}{\partial R_{At}} = \sum_M \frac{\partial P_{M g}}{\partial R_{At}}
$$

In [26]:
dR_Z = dR_P.sum(axis=0)

我们也可以获取格点所在原子 Becke 权重的导数 `dR_Pg`，维度 $(A, t, g)$ `(natm, 3, ngrids)`：

$$
\frac{\partial P_g^M}{\partial R_{At}} = \frac{\partial P_{M g}}{\partial R_{At}} \delta_{g \in M}
$$

不过这里有需要当心的地方。NumPy 的 advanced indexing 的维度可能与直观上认为的不同，必要情况下需要先检查，然后作转置。在 C/Rust 程序或张量框架中，不支持如此复杂的 advanced indexing，因此实际情景下是手动索引。

In [27]:
dR_Pg = dR_P[atm_indices, :, :, np.arange(ngrids)].transpose(1, 2, 0)

### 格点权重 $w_g$ 导数

我们首先在不考虑格点坐标梯度的前提下，给出格点权重 $w_g$ 的导数。回顾格点权重定义：

$$
w_g = w_g^\text{quad} \frac{P^M_g}{Z_g}
$$

其导数 `dw`，维度 $(A, t, g)$ `(natm, 3, ngrids)`，可以依简单的链式法则给出：

$$
\frac{\partial w_g}{\partial R_{At}} = w_g^\text{quad} \left( \frac{1}{Z_g} \frac{\partial P^M_g}{\partial R_{At}} - \frac{P^M_g}{Z_g^2} \frac{\partial Z_g}{\partial R_{At}} \right)
$$

In [28]:
dw = wquad * (dR_Pg / Z - Pg / Z**2 * dR_Z)
dw[atm_indices, :, np.arange(ngrids)] = 0.0
dw[atm_indices, :, np.arange(ngrids)] = -dw.sum(axis=0).T

In [29]:
np.allclose(dw, dw_ref[:, :, nonpad_mask])

True

### 对于缩并中间量的优化

这里特指对下述项的处理：

$$
\frac{\partial P_{M g}}{\partial R_{A t}}
$$

该项是一个 4-dim 张量 $(M, A, t, g)$ `(natm, natm, 3, ngrids)`。从计算量上这是省不下来的；但从存储上，我们实际上只在 $w_g$ 导数上应用了这两个表达式：

$$
\frac{\partial P^M_g}{\partial R_{At}}, \; \frac{\partial Z_g}{\partial R_{At}}
$$

这两个表达式都是 3-dim 张量 $(A, t, g)$ `(natm, 3, ngrids)`。因此在计算时，我们应该想办法直接缩并到 3-dim 张量，而不是先计算 4-dim 张量再缩并。